# 02 - Built-in On-Demand Evaluations

This notebook shows how to review the quality of a completed CityAnalyst session. AgentCore evaluates the trace that was recorded when the agent ran, so you can ask focused questions about the answer and tool use without deploying the agent again.

You will:

1. Choose an evaluator based on the question you want to answer.
2. Evaluate one recorded session with the AgentCore CLI.
3. Read the scores, labels, and explanations in a clear results view.
4. Run the same kind of evaluation from Python with `EvaluationClient`.

**Estimated time:** 30-45 minutes  
**Creates AWS resources:** No persistent resources. Evaluation calls use a judge model and can incur cost.

## 1. Choose what you want to measure

A built-in evaluator is a ready-to-use reviewer for one part of agent behavior. Before choosing one, decide how much of the recorded activity it needs to inspect:

- **SESSION** looks at the full interaction and is useful for questions such as, "Did the agent finish the user's task?"
- **TRACE** looks at one user request and agent response. Most response-quality checks use this level.
- **TOOL_CALL** looks at one tool action, including the selected tool and its inputs.

AgentCore CLI `0.27.0` includes the following built-in evaluators:

| Evaluator | Level | Use it to ask |
|---|---|---|
| `Builtin.GoalSuccessRate` | SESSION | Did the conversation complete the user's goal? |
| `Builtin.Correctness` | TRACE | Is the response correct, especially against a reference? |
| `Builtin.Faithfulness` | TRACE | Is the response grounded in supplied context? |
| `Builtin.Helpfulness` | TRACE | Is the response useful? |
| `Builtin.ResponseRelevance` | TRACE | Does the response address the request? |
| `Builtin.Conciseness` | TRACE | Is the response appropriately concise? |
| `Builtin.Coherence` | TRACE | Is the response logically coherent? |
| `Builtin.InstructionFollowing` | TRACE | Did the agent follow its instructions? |
| `Builtin.Refusal` | TRACE | Did the agent handle refusal behavior appropriately? |
| `Builtin.ToolSelectionAccuracy` | TOOL_CALL | Was this the right tool for the task? |

The catalog can grow as AgentCore adds evaluators. If you want to confirm the IDs supported by your installed CLI, run `agentcore run eval --help`.

## 2. Create a session to evaluate

An on-demand evaluation reviews the trace data recorded during a completed agent run. This recorded data is also called telemetry. For this exercise, we will invoke CityAnalyst once, wait for its trace to arrive, and then evaluate that session. Trace ingestion can take a few seconds, so the helper waits until the session is available.

In [ ]:
import json

from IPython.display import Markdown, display

from src.workshop_utils import (
    GENERATED_DIR,
    RUNTIME_NAME,
    make_session_id,
    run_cli_json,
    wait_for_session_trace,
)

GENERATED_DIR.mkdir(exist_ok=True)
SESSION_ID = make_session_id("builtins")
PROMPT = "What are the population and land area of Seattle, WA?"

invocation = run_cli_json(
    "invoke",
    "--runtime",
    RUNTIME_NAME,
    "--session-id",
    SESSION_ID,
    "--prompt",
    PROMPT,
)
agent_response = invocation.get("response", invocation)
if isinstance(agent_response, str):
    try:
        agent_response = json.loads(agent_response)
    except json.JSONDecodeError:
        pass

display(Markdown(f"**Prompt:** {PROMPT}"))
display(agent_response)

trace = wait_for_session_trace(SESSION_ID)
print(f"Session ready for evaluation: {SESSION_ID}")
print(f"Trace: {trace['traceId']}")

## 3. Evaluate the session with the CLI

Reference inputs describe what a good result should contain or do. They help the evaluator judge meaning and behavior without requiring the agent to use exactly the same wording.

- **`expected_response`** gives an example of the facts or meaning expected in the answer. `Correctness` can compare the agent's response with this example even when the wording differs.
- **`assertions`** state requirements in plain language, such as identifying Seattle and using the workshop data.
- **`expected_trajectory`** lists the tool calls expected for the task, in order.

Each evaluator uses the information that matches its job. For example, `Correctness` uses the expected response, `GoalSuccessRate` can use the assertions, and `ToolSelectionAccuracy` compares the recorded tool calls with the expected trajectory.

In [ ]:
cli_result = run_cli_json(
    "run",
    "eval",
    "--runtime",
    RUNTIME_NAME,
    "--session-id",
    SESSION_ID,
    "--evaluator",
    "Builtin.GoalSuccessRate",
    "Builtin.Correctness",
    "Builtin.Helpfulness",
    "Builtin.InstructionFollowing",
    "Builtin.ToolSelectionAccuracy",
    "--expected-response",
    (
        "Seattle, WA has population 780995 and land area "
        "83.8 square miles in the workshop dataset."
    ),
    "--assertion",
    "The response must identify Seattle, WA and use the workshop facts.",
    "--expected-trajectory",
    "lookup_city",
)

cli_run = cli_result.get("run", cli_result)
print(f"Evaluation completed for {cli_run.get('sessionCount', 0)} session(s).")
if cli_result.get("filePath"):
    print(f"Raw result saved to: {cli_result['filePath']}")

### Read the CLI results

The CLI returns nested JSON because one run can contain several evaluators, and each evaluator can score several targets. The next cell reshapes that JSON for display; the original response remains available in `cli_result`.

Start with the summary table to compare evaluators. The **aggregate score** is the average for all matching targets in this run. A session evaluator usually produces one target, while a tool-call evaluator can produce a result for each recorded tool call. **Judge tokens** show how many model tokens that evaluator used.

After the table, read each explanation. The explanation is often more useful than the number because it tells you what the evaluator noticed and what could be improved.

In [ ]:
import pandas as pd

def format_score(value):
    return "Not available" if value is None else f"{float(value):.2f}"

def target_details(result):
    if result.get("spanId"):
        return "Tool call", result["spanId"]
    if result.get("traceId"):
        return "Trace", result["traceId"]
    return "Session", result.get("sessionId", "Not available")

def show_result_explanations(rows):
    if not rows:
        print("No target-level evaluation results were returned.")
        return

    current_evaluator = None
    for row in rows:
        evaluator = row["evaluator"] or "Unknown evaluator"
        if evaluator != current_evaluator:
            current_evaluator = evaluator
            display(Markdown(f"### {current_evaluator.removeprefix('Builtin.')}"))

        message = row["error"] or row["explanation"] or "No explanation was returned."
        message_label = "Error" if row["error"] else "Explanation"
        display(
            Markdown(
                f"**{row['target_type']}:** `{row['target_id']}`  \n"
                f"**Score:** `{format_score(row['value'])}` | "
                f"**Label:** `{row['label'] or 'Not available'}`\n\n"
                f"**{message_label}:** {message}"
            )
        )

cli_summary_rows = []
cli_detail_rows = []

for evaluator_result in cli_run.get("results", []):
    scores = evaluator_result.get("sessionScores", [])
    errors = [score for score in scores if score.get("errorCode") or score.get("errorMessage")]
    token_usage = evaluator_result.get("tokenUsage", {})
    cli_summary_rows.append(
        {
            "Evaluator": evaluator_result.get("evaluator", "Unknown"),
            "Aggregate score": evaluator_result.get("aggregateScore"),
            "Targets scored": len(scores),
            "Errors": len(errors),
            "Judge tokens": token_usage.get("totalTokens"),
        }
    )

    for score in scores:
        target_type, target_id = target_details(score)
        cli_detail_rows.append(
            {
                "evaluator": evaluator_result.get("evaluator", "Unknown"),
                "target_type": target_type,
                "target_id": target_id,
                "value": score.get("value"),
                "label": score.get("label"),
                "explanation": score.get("explanation"),
                "error": score.get("errorMessage") or score.get("errorCode"),
            }
        )

display(pd.DataFrame(cli_summary_rows))
show_result_explanations(cli_detail_rows)

The CLI also saves each run under `agentcore/.cli/eval-results/`. This history is useful when you want to compare recent runs without repeating the evaluation. The next cell shows one row per evaluator from the ten most recent runs.

The raw history remains available in the `history` variable if you want to inspect the complete JSON.

In [ ]:
history = run_cli_json("evals", "history", "--limit", "10")

history_rows = []
for run in history.get("runs", []):
    for evaluator_result in run.get("results", []):
        history_rows.append(
            {
                "Run time": run.get("timestamp"),
                "Agent": run.get("agent"),
                "Evaluator": evaluator_result.get("evaluator"),
                "Aggregate score": evaluator_result.get("aggregateScore"),
                "Sessions": run.get("sessionCount"),
            }
        )

if history_rows:
    display(pd.DataFrame(history_rows))
else:
    print("No saved evaluation runs were found.")

## 4. Python automation with `EvaluationClient`

`EvaluationClient` is a convenience helper in the AgentCore Python SDK. It is another interface to the same AgentCore evaluation service used by the CLI, not a different scoring system. You provide a session ID and evaluator IDs, and it handles the work of finding spans and building evaluation requests for you:

1. It finds the session's spans in CloudWatch.
2. It checks whether each evaluator works at the session, trace, or tool-call level.
3. It builds the correct evaluation targets.
4. It calls the AgentCore `Evaluate` API and returns the results as Python dictionaries.

Use the **CLI** for a quick investigation, a terminal workflow, or a small shell-based quality check. Use **`EvaluationClient`** when Python needs to choose evaluators dynamically, loop over sessions, combine results with other data, or feed evaluation output into a test or reporting pipeline.

`ReferenceInputs` is the Python form of the expected response, assertions, and expected trajectory used in the CLI example. `look_back_time` tells the client how far back in CloudWatch to search for the session.

`EvaluationClient` evaluates recorded spans; it does not invoke the agent. Notebook 03 introduces a dataset runner for workflows that need to invoke the agent across many scenarios.

In [ ]:
from datetime import timedelta

from bedrock_agentcore.evaluation import EvaluationClient, ReferenceInputs
from src.workshop_utils import load_runtime_info

runtime = load_runtime_info()
evaluation_client = EvaluationClient(region_name=runtime.region)

# Provide the same reference information used in the CLI example.
references = ReferenceInputs(
    expected_response=(
        "Seattle, WA has population 780995 and land area "
        "83.8 square miles in the workshop dataset."
    ),
    assertions=[
        "The answer identifies Seattle, WA.",
        "The answer uses the fixed workshop facts.",
    ],
    expected_trajectory=["lookup_city"],
)

sdk_results = evaluation_client.run(
    evaluator_ids=[
        "Builtin.GoalSuccessRate",
        "Builtin.Correctness",
        "Builtin.Helpfulness",
        "Builtin.ToolSelectionAccuracy",
    ],
    session_id=SESSION_ID,
    agent_id=runtime.runtime_id,
    look_back_time=timedelta(hours=1),  # Search recent CloudWatch spans.
    reference_inputs=references,
)

print(f"EvaluationClient returned {len(sdk_results)} target-level result(s).")

## 5. Normalize the response for analysis

The SDK returns a list of dictionaries that closely follows the AgentCore API response. The exact context fields depend on what was evaluated: a session result has a session ID, a trace result also has a trace ID, and a tool-call result also has a span ID.

**Normalizing** means turning those slightly different dictionaries into rows with the same columns. For example, every row below will have an evaluator, target type, target ID, score, label, explanation, and error field. This makes the results easier to compare, filter, chart, or save to a file.

Normalization does not change the scores or explanations. The original SDK response remains available in `sdk_results`; `results_df` is simply a more convenient view for analysis.

In [ ]:
def normalize_sdk_results(results):
    rows = []
    for item in results:
        context = item.get("context", {})
        if context.get("spanId"):
            target_type, target_id = "Tool call", context["spanId"]
        elif context.get("traceId"):
            target_type, target_id = "Trace", context["traceId"]
        else:
            target_type = "Session"
            target_id = context.get("sessionId", SESSION_ID)

        rows.append(
            {
                "evaluator": item.get("evaluatorId") or "Unknown evaluator",
                "target_type": target_type,
                "target_id": target_id,
                "value": item.get("value"),
                "label": item.get("label"),
                "explanation": item.get("explanation"),
                "error": item.get("errorMessage") or item.get("errorCode"),
            }
        )
    return rows

normalized_results = normalize_sdk_results(sdk_results)
results_df = pd.DataFrame(normalized_results)

display(
    results_df[
        ["evaluator", "target_type", "target_id", "value", "label", "error"]
    ]
)
show_result_explanations(normalized_results)

## 6. Read the result as a whole

A result contains three complementary views of the same judgment:

- **`value`** is the numeric score, usually between 0 and 1. It is useful for summaries and comparisons.
- **`label`** translates the score into a category such as `Correct`, `Very Helpful`, or `Yes`.
- **`explanation`** describes what the evaluator observed. Read this when you want to understand a score or decide what to improve.

A score should not become an automatic pass or fail rule until you have tested how the evaluator behaves on your own examples. A value of `0.8` may be strong for one evaluator and insufficient for another.

To choose a useful threshold:

1. Collect examples that represent real successes and failures.
2. Have people review and label those examples.
3. Run the evaluator and compare its decisions with the human labels.
4. Choose a threshold that supports a specific action, such as blocking a release or opening a review ticket.
5. Recheck the threshold as the agent, evaluator, and traffic change.

## 7. Evaluator selection exercise

Start with the quality question you need to answer, then choose the smallest evaluator set that can answer it:

| Suspected failure | Start with |
|---|---|
| Correct facts, but the task is unfinished | `GoalSuccessRate` |
| Wrong city values | `Correctness` with an expected response |
| Correct answer, wrong or unnecessary tool | `ToolSelectionAccuracy` |
| Answer ignores the required response format | `InstructionFollowing` and a custom schema evaluator |
| Answer includes claims that are not supported by retrieved context | `Faithfulness` |

More evaluators do not automatically produce better insight. Each evaluator adds model usage and another signal to interpret, so choose evaluators that connect to a real failure you want to detect.

## 8. Checkpoint

You now have two ways to request an on-demand evaluation:

- Use the **CLI** for quick, targeted inspection of recorded sessions.
- Use **`EvaluationClient`** when evaluation is part of Python automation or analysis.

Both call the AgentCore evaluation service. Because an LLM-based evaluator makes a new judgment on each run, small score or wording differences are possible even when the inputs are unchanged.

Continue to [03 - Ground Truth and Curated Datasets](03-ground-truth-and-datasets.ipynb) to turn one-session inspection into a repeatable regression suite.